In [1]:
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime, timezone


class Entity(BaseModel):
    name: str = Field(..., description="The name of the entity (e.g., 'Sarah', 'Python', 'Project Apollo')")
    entity_type: str = Field(..., description="The category of the entity (e.g., 'Person', 'Technology', 'Project', 'Location')")

class AtomicNote(BaseModel):
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc), description="When the memory was captured")
    content: str = Field(..., description="A clean, concise, standalone summary of the input.")
    # UPDATED: Flexible source_type
    source_type: str = Field(..., description="Infer the type of data this is (e.g., 'fleeting_thought', 'code_snippet', 'meeting_note', 'grocery_list', 'book_quote'). Use short, snake_case strings.")
    # UPDATED: Flexible domain
    domain: str = Field(..., description="The broad life category this note belongs to. Infer this dynamically (e.g., 'PhD Research', 'Personal', 'Home Improvement'). Keep it high-level.")
    entities: List[Entity] = Field(default_factory=list, description="Specific nouns, people, projects, or technologies mentioned.")
    tags: List[str] = Field(default_factory=list, description="General keywords for easy filtering (e.g., ['gift-idea', 'bug-fix'])")

In [ ]:
from typing import List

class IngestionPayload(BaseModel):
    notes: List[AtomicNote] = Field (description="A list of distinct, atomic thoughts or data points extracted from the input.")

# Inside your KnowledgeSynthesizer class:
async def extract_and_save(self, raw_text: str):
    # The LLM now looks for multiple 'atoms' in the input
    structured_llm = self.llm.with_structured_output(IngestionPayload)
    payload = await structured_llm.ainvoke(
        f"Decompose the following input into distinct atomic memories. "
        f"If the input contains multiple unrelated ideas, create a separate note for each: {raw_text}"
    )

    for note in payload.notes:
        # Save each 'atom' individually
        await self._save_to_vector(note)
        await self._sync_to_graph(note)
    
    return payload.notes

In [3]:
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
      # fast + good for testing
)
checkpointer = InMemorySaver()


NameError: name 'ChatGroq' is not defined

In [2]:
from langchain_core.prompts import ChatPromptTemplate

def extract_memory(raw_text: str) -> AtomicNote:
    # 1. Bind the Pydantic schema to Groq
    structured_llm = llm.with_structured_output(AtomicNote)
    
    # 2. Give the LLM its instructions
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an AI cognitive extractor. Your job is to take messy, unstructured user input and convert it into a clean, structured Atomic Note."),
        ("user", "{input_text}")
    ])
    
    # 3. Chain and invoke
    chain = prompt | structured_llm
    result = chain.invoke({"input_text": raw_text})
    
    # 4. Guarantee the original text is saved exactly as typed
    result.original_raw_text = raw_text 
    
    return result

# --- TEST IT ---
sample_input = """
Just finished the weekly sync with Sarah. 
She mentioned that the Redis cache is causing memory leaks in the 'Project Apollo' production environment. 
Need to check the AWS logs on Monday. Also, remind me to buy dog food on the way home.
"""

memory = extract_memory(sample_input)
print(memory.model_dump_json(indent=2))

/home/awabzem/Documents/rt/pfa/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'llm' is not defined